In [1]:
from VerifyObjects import *
from VerifyRelation import *
from Verifier import *

In [2]:
from IPython.display import display, Math, Markdown

def printmd(veri_ob):
    display(Markdown("$\\displaystyle %s$" % repr(veri_ob)))

def display_prop():
    for p in Prop:
        display(p)

# Norm

In [3]:
class Norm(Scalar):
    def __init__(self, inside):
        self.inside = inside
        self.base = self
        self.is_nonnegative = 1

    def __pow__(self, exponent):
        if exponent==2:
            return NS(self.inside)
        else:
            return super().__pow__(exponent)
        
    def __str__(self):
        s = "|| " + str(self.inside) + " ||"
        return s

    # def __eq__(self, other):
    #     return isinstance(other, Norm) and ( self.inside == other.inside or self.inside == (-other).inside )

    # xxx
    # def __hash__(self):
    #     return hash( str(type(self)) + str(self.inside) )

In [4]:
v = Vector('v')

nv = Norm(v)
display(nv)
display(NS(v)==nv**2)

|| v ||

True

# Lemma 1

## Definition of materials

In [5]:
Prop.clear()

In [6]:
x = Sequence('x')
lamda = ScalarSequence('\lambda')
theta = ScalarSequence('\\theta')
k = IterationCount('k')
f = Function('f')
g = Gradient(f)
x_star = x('\star')
f_star = f(x('\star'))

## First equation by definition of $x$

In [7]:
def eq1_1(k): return Equal( NS(x(k+1) -  x_star), NS(x(k) - x_star) + 2 * IP( x(k+1) - x(k), x(k) - x_star ) + NS(x(k+1) - x(k)) )
def eq1_2(k): return Equal( eq1_1(k).rhs , NS(x(k) - x_star) + 2 *  IP( lamda(k) * g(x(k)), x_star - x(k) ) + NS(x(k+1) - x(k)))

display(eq1_1(k))
display(eq1_2(k))


|| -x^{\star} + x^{1+k} ||^2 = 2  < -x^{\star} + x^{k}, -x^{k} + x^{1+k} >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

2  < -x^{\star} + x^{k}, -x^{k} + x^{1+k} >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2 = 2  < -x^{k} + x^{\star}, \lambda_{k} ▽f(x^{k}) >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

In [8]:
verify(eq1_1(k), [] )

True

In [9]:
def def_x(n): return Equal( x(n), x(n-1) - lamda(n-1) * g(x(n-1)) )

display(def_x(k))

x^{k} = -\lambda_{-1+k} ▽f(x^{-1+k}) + x^{-1+k}

In [10]:
propadd_assumption(def_x)
propadd_intro(def_x, k+1)

In [11]:
verify(eq1_2(k), set( [ def_x(k+1) ] )) 

True

## Ineq6 by convexity of $f$

In [12]:
def ineq6_1(k): return ( IP( g(x(k)) , x_star - x(k) ) <= f(x_star) - f(x(k)) )
def ineq6(k): return  2*lamda(k)*ineq6_1(k).lhs <=  2*lamda(k)*ineq6_1(k).rhs

display(ineq6_1(k))
display(ineq6(k))

\langle -x^{k} + x^{\star}, ▽f(x^{k})\rangle \le - f(x^{k})+f(x^{\star})

2\lambda_{k}\langle -x^{k} + x^{\star}, ▽f(x^{k})\rangle \le 2\left( - f(x^{k})+f(x^{\star}) \right)\lambda_{k}

### Convexity inequality

In [13]:
# f.is_convex = True
f = Function('f', convex=True)

def convex_ineq(ft,v,w): 
    if ft.is_convex == True:
        return IP( Gradient(ft)(v) , w-v) <= ft(w) - ft(v)
    else:
        raise TypeError( str(ft) + 'is not convex')

In [64]:
v = Vector('v')
w = Vector('w')
convex_ineq(f,v,w)
convex_ineq(f,x(k),x_star)

\langle -x^{k} + x^{\star}, ▽f(x^{k})\rangle \le - f(x^{k})+f(x^{\star})

In [15]:
propadd_assumption(convex_ineq)
propadd_intro(convex_ineq, (f,x(k),x_star) )

In [16]:
propadd_assumption( 0 <= lamda(k) ) ## Assumption
mul_inequality( 0 <= lamda(k) , 2)
mul_inequality( ineq6_1(k) , 2 * lamda(k) )

2\lambda_{k}\langle -x^{k} + x^{\star}, ▽f(x^{k})\rangle \le 2\left( - f(x^{k})+f(x^{\star}) \right)\lambda_{k}

In [17]:
verify( ineq6(k), [] )

True

## Ineq7 - Concaternating above results

In [18]:
def ineq7(k): return ( NS( x(k+1) - x_star ) <= NS( x(k) - x_star ) - 2 * lamda(k) * ( f(x(k)) - f_star ) + NS( x(k+1) - x(k) ) )

display( ineq7(k) )

|| -x^{\star} + x^{1+k} ||^2 \le -2 ( - f(x^{\star})+f(x^{k}) ) \lambda_{k}+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

In [19]:
sum_of_nonneg( NS( x(k) - x_star ), NS( x(k+1) - x(k) ) )

def ineq7_1(k): return add_of( ineq6(k) , NS( x(k) - x_star ) + NS( x(k+1) - x(k) ))

In [20]:
display( eq1_1(k) )
display( eq1_2(k) )
display( ineq7_1(k) )
print( [eq1_1(k).rhs == eq1_2(k).lhs, 
        eq1_2(k).rhs.simplify() == ineq7_1(k).lhs.simplify()] )

|| -x^{\star} + x^{1+k} ||^2 = 2  < -x^{\star} + x^{k}, -x^{k} + x^{1+k} >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

2  < -x^{\star} + x^{k}, -x^{k} + x^{1+k} >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2 = 2  < -x^{k} + x^{\star}, \lambda_{k} ▽f(x^{k}) >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

2 \lambda_{k} < -x^{k} + x^{\star}, ▽f(x^{k}) >+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2 \le 2 ( - f(x^{k})+f(x^{\star}) ) \lambda_{k}+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

[True, True]


### New concaternating method

In [21]:
propadd_repeat_transivity_eq_le_lt( [ eq1_1(k), eq1_2(k), ineq7_1(k) ] )

|| -x^{\star} + x^{1+k} ||^2 \le 2 ( - f(x^{k})+f(x^{\star}) ) \lambda_{k}+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

In [22]:
verify( ineq7(k), [] )

True

## Eq8

In [23]:
def eq8_1(k): return Equal( NS( x(k+1) - x(k) ) , 2 * NS( x(k+1) - x(k) ) - NS( x(k+1) - x(k) ) )
def eq8_2(k): return Equal( 2 * NS( x(k+1) - x(k) ) - NS( x(k+1) - x(k) ), -2 * lamda(k) * IP( g(x(k)) , x(k+1) - x(k)  ) - NS( x(k+1) - x(k) ) )
def eq8_3(k): return Equal(  -2 * lamda(k) * IP( g(x(k)) , x(k+1) - x(k)  ) - NS( x(k+1) - x(k) ) ,
            2 * lamda(k) * IP( g(x(k)) - g(x(k-1)) , x(k) - x(k+1)  ) + 2 * lamda(k) * IP( g(x(k-1)) , x(k) - x(k+1)  ) - NS( x(k+1) - x(k) )  )

In [24]:
print( verify(eq8_1(k), set() ) )
print( verify(eq8_2(k), set([ def_x(k+1) ] ) ) )
print( verify(eq8_3(k), set() ) )

True
True
True


## Ineq9

In [25]:
def ineq9_1(k): return 2 * lamda(k) * IP( g(x(k)) - g(x(k-1)) , x(k) - x(k+1)  ) <= 2 * lamda(k) * Norm( g(x(k)) - g(x(k-1)) ) * Norm( x(k) - x(k+1) )
def ineq9_2(k): return 2 * lamda(k) * Norm( g(x(k)) - g(x(k-1)) ) * Norm( x(k) - x(k+1) )  <= Norm( x(k) - x(k-1) ) * Norm( x(k) - x(k+1) )
def ineq9_3(k): return Norm( x(k) - x(k-1) ) * Norm( x(k) - x(k+1) ) <= 1/2 * NS( x(k) - x(k-1) ) + 1/2 * NS( x(k) - x(k+1) )

display( ineq9_1(k) )
display( ineq9_2(k) )
display( ineq9_3(k) )

2\lambda_{k}\langle -x^{1+k} + x^{k}, -▽f(x^{-1+k}) + ▽f(x^{k})\rangle \le 2\lambda_{k}|| -x^{1+k} + x^{k} |||| -▽f(x^{-1+k}) + ▽f(x^{k}) ||

2\lambda_{k}|| -x^{1+k} + x^{k} |||| -▽f(x^{-1+k}) + ▽f(x^{k}) || \le || -x^{-1+k} + x^{k} |||| -x^{1+k} + x^{k} ||

|| -x^{-1+k} + x^{k} |||| -x^{1+k} + x^{k} || \le 0.5 || -x^{-1+k} + x^{k} ||^2+0.5 || -x^{1+k} + x^{k} ||^2

### ineq9_1, by CS inequality

In [26]:
def CS_ineq(v,w): 
    if v.is_vector and w.is_vector:
        return IP( v, w ) <= Norm(v) * Norm( w )
    else:
        raise TypeError('Cauchy Schwarz inequaltiy holds for vectors')

In [66]:
propadd_assumption( CS_ineq )
propadd_intro( CS_ineq, ( g(x(k)) - g(x(k-1)) , x(k) - x(k+1) ) )
mul_inequality( CS_ineq( g(x(k)) - g(x(k-1)) , x(k) - x(k+1) ) , 2 * lamda(k)  )

verify( ineq9_1(k), [] )

True

In [ ]:
"""
step1 = propadd_assumption( CS_ineq )
step2 = propadd_intro( CS_ineq, ( g(x(k)) - g(x(k-1)) , x(k) - x(k+1) ) )
step3 = mul_inequality( CS_ineq( g(x(k)) - g(x(k-1)) , x(k) - x(k+1) ) , 2 * lamda(k)  ) # + 2 * lamda(k) 의 positiveness도 유저가 제시

verify( ineq9_1(k), [step1, step2, step3] )
"""

### Ineq9_2, by definition of $\lambda_k$

### Min

In [28]:
def is_iterable(obj):
    try:
        iter(obj)
        return True
    except TypeError:
        return False

In [29]:
class Min(Scalar):
    def __init__(self, min_set):
        self.base = self
        self.min_set = min_set
        if is_iterable(min_set):
            for element in min_set:
                Prop.add( le(self, element) )

    def __str__(self):
        s = 'Min('
        if is_iterable(self.min_set):
            for element in self.min_set:
                s += str(element) + ', '
            s = s[:-2]
        s += ')'
        return s

    def __repr__(self):
        s = '\\text{Min} \left('
        if is_iterable(self.min_set):
            for element in self.min_set:
                s += repr(element) + ', \,\,'
            s = s[:-6]
        s += '\\right)'
        return s

In [30]:
a = Scalar('a')
b = Scalar('b')

min_set = [a,b]

m = Min(min_set)

In [31]:
verify(m<=a, set())

True

### Define $\lambda_k$ with Min

In [32]:
def def_lambda(n): return Equal( lamda(n), Min( [ Sqrt(1+theta(n-1)) * lamda(n-1) , Norm(x(n) - x(n-1)) / ( 2 * Norm( g(x(n)) - g(x(n-1)) ) ) ] ) )

display(def_lambda(k))

\lambda_{k} = \text{Min} \left(\sqrt{ 1+\theta_{-1+k}}\lambda_{-1+k}, \,\,0.5|| -x^{-1+k} + x^{k} |||| -▽f(x^{-1+k}) + ▽f(x^{k}) ||^{-1}\right)

In [33]:
propadd_assumption(def_lambda)
propadd_intro(def_lambda, k)
propadd_transivity( def_lambda(k) , def_lambda(k).rhs <= Norm(x(k) - x(k-1)) / ( 2 * Norm( g(x(k)) - g(x(k-1)) ) ) )
mul_inequality( lamda(k)  <= Norm(x(k) - x(k-1)) / ( 2 * Norm( g(x(k)) - g(x(k-1)) ) ), 2 * Norm( g(x(k)) - g(x(k-1)) ) * Norm(x(k) - x(k+1)) )

2\lambda_{k}|| -x^{1+k} + x^{k} |||| -▽f(x^{-1+k}) + ▽f(x^{k}) || \le || -x^{-1+k} + x^{k} |||| -x^{1+k} + x^{k} ||

In [34]:
verify( ineq9_2(k) , [] )

True

#### Testing is_nonnegative

In [35]:
print( ( 2 * Norm( g(x(k)) - g(x(k-1)) ) * Norm(x(k) - x(k+1)) ).is_nonnegative )

1


In [36]:
a = Scalar('a')
a.is_nonnegative = 1

print( (a+a+4).is_nonnegative )

1


## ineq9_3 by Young's inequality

In [37]:
def Youngs_ineq(a,v,w):
    if type(v) != type(w):
        raise TypeError('Input proper types for Youngs inequaltiy')
    if is_scalar_or_number(a) and v.is_vector and w.is_vector:
        return IP( v, w ) <= 1/(2*a**2) * NS(v) + a**2/2 * NS( w )
    elif is_scalar_or_number(a) and is_scalar_or_number(v) and is_scalar_or_number(w):
        return v * w <= 1/(2*a**2) * v**2 + a**2/2 * w**2
    else:
        raise TypeError('Input proper types for Youngs inequaltiy')

In [38]:
propadd_assumption( Youngs_ineq )
propadd_intro( Youngs_ineq, ( 1, Norm( x(k) - x(k-1) ), Norm( x(k) - x(k+1) ) ) )

In [39]:
verify( ineq9_3(k), [] )

True

# Ineq10

In [40]:
def eq10_1(k): return Equal( 2 * lamda(k) * IP( g(x(k-1)), x(k) - x(k+1) ) , 2*lamda(k)/lamda(k-1) * IP( x(k-1) - x(k), x(k) - x(k+1) )  )
def eq10_2(k): return Equal( eq10_1(k).rhs, 2 * lamda(k) * theta(k) * IP( x(k-1) - x(k) , g(x(k)) ) )
def ineq10_3(k): return eq10_2(k).rhs <= 2 * lamda(k) * theta(k) * ( f(x(k-1)) - f(x(k)) )

### eq10_1 by definition of $x_k$

In [41]:
propadd_intro(def_x, k)

verify( eq10_1(k) , [ def_x(k) ] )

True

### eq10_2 by definition of $\theta_k$ and $x_{k+1}$

In [42]:
def def_theta(k): return Equal( theta(k) , lamda(k)/lamda(k-1) )

In [43]:
propadd_assumption(def_theta)
propadd_intro(def_theta, k)

In [44]:
verify( eq10_2(k) , [ def_theta(k), def_x(k+1) ] )

True

### ineq10_3 by convexity of $f$ and nonnegativity of $\theta$

In [45]:
propadd_intro( convex_ineq, (f, x(k), x(k-1)) )

In [63]:
propadd_assumption( 0 <= theta(k) )  ### added without proof
mul_inequality( 0 <= theta(k), 2 ) 
mul_inequality( 0 <= lamda(k), 2 * theta(k) ) 
verify( 0 <= 2 * lamda(k) * theta(k), [] )

mul_inequality( convex_ineq(f, x(k), x(k-1)) , 2 * lamda(k) * theta(k) )

2\lambda_{k}\theta_{k}\langle -x^{k} + x^{-1+k}, ▽f(x^{k})\rangle \le 2\left( - f(x^{k})+f(x^{-1+k}) \right)\lambda_{k}\theta_{k}

In [47]:
verify( ineq10_3(k) , [ mul_inequality( convex_ineq(f, x(k), x(k-1)) , 2 * lamda(k) * theta(k) ) ] )

True

#### $ \left\| x^{k+1} - x^k \right\|^2 \le \frac{1}{2} \left\| x^{k} - x^{k-1} \right\|^2 - \frac{1}{2} \left\| x^{k+1} - x^k \right\|^2 + 2 \lambda_k \theta_k ( f(x^{k-1}) - f(x^k) )   $
by plugging ineq9, ineq10 to eq8

In [48]:
def ineq2(k): return NS( x(k+1) - x(k) ) <= 1/2 * NS( x(k) - x(k-1) ) - 1/2 * NS( x(k+1) - x(k) ) + 2 * lamda(k) * theta(k) * ( f(x(k-1)) - f(x(k)) )

display( ineq2(k) )

|| -x^{k} + x^{1+k} ||^2 \le -0.5 || -x^{k} + x^{1+k} ||^2+0.5 || -x^{-1+k} + x^{k} ||^2+2 ( - f(x^{k})+f(x^{-1+k}) ) \lambda_{k} \theta_{k}

In [49]:
def eq8(k): return propadd_repeat_transivity_eq_le_lt( [ eq8_1(k), eq8_2(k), eq8_3(k)  ] )
def ineq9(k): return propadd_repeat_transivity_eq_le_lt( [ ineq9_1(k), ineq9_2(k), ineq9_3(k)  ] )
def ineq10(k): return propadd_repeat_transivity_eq_le_lt( [ eq10_1(k), eq10_2(k), ineq10_3(k)  ] )

display( eq8(k) )
display( ineq9(k) )
display( ineq10(k) )

|| -x^{k} + x^{1+k} ||^2 = -|| -x^{k} + x^{1+k} ||^2+2 \lambda_{k} < -x^{1+k} + x^{k}, -▽f(x^{-1+k}) + ▽f(x^{k}) >+2 \lambda_{k} < -x^{1+k} + x^{k}, ▽f(x^{-1+k}) >

2\lambda_{k}\langle -x^{1+k} + x^{k}, -▽f(x^{-1+k}) + ▽f(x^{k})\rangle \le 0.5 || -x^{-1+k} + x^{k} ||^2+0.5 || -x^{1+k} + x^{k} ||^2

2\lambda_{k}\langle -x^{1+k} + x^{k}, ▽f(x^{-1+k})\rangle \le 2\left( - f(x^{k})+f(x^{-1+k}) \right)\lambda_{k}\theta_{k}

In [50]:
propadd_le_ge_to_nonneg( ineq9(k) )
propadd_le_ge_to_nonneg( ineq10(k) )

propadd_sum_of_nonneg( ineq9(k).rhs - ineq9(k).lhs, ineq10(k).rhs - ineq10(k).lhs )

0 \le -2 \lambda_{k} < -x^{1+k} + x^{k}, -▽f(x^{-1+k}) + ▽f(x^{k}) >-2 \lambda_{k} < -x^{1+k} + x^{k}, ▽f(x^{-1+k}) >+0.5 || -x^{-1+k} + x^{k} ||^2+0.5 || -x^{1+k} + x^{k} ||^2+2 ( - f(x^{k})+f(x^{-1+k}) ) \lambda_{k} \theta_{k}

In [51]:
add_of( 0 <= ( ineq9(k).rhs - ineq9(k).lhs ) + (ineq10(k).rhs - ineq10(k).lhs), eq8(k).rhs  )

-|| -x^{k} + x^{1+k} ||^2+2 \lambda_{k} < -x^{1+k} + x^{k}, -▽f(x^{-1+k}) + ▽f(x^{k}) >+2 \lambda_{k} < -x^{1+k} + x^{k}, ▽f(x^{-1+k}) > \le -|| -x^{k} + x^{1+k} ||^2+0+0+0.5 || -x^{-1+k} + x^{k} ||^2+0.5 || -x^{1+k} + x^{k} ||^2+2 ( - f(x^{k})+f(x^{-1+k}) ) \lambda_{k} \theta_{k}

In [52]:
propadd_transivity( eq8(k), add_of( 0 <= ( ineq9(k).rhs - ineq9(k).lhs ) + (ineq10(k).rhs - ineq10(k).lhs), eq8(k).rhs  ) )

In [53]:
verify( ineq2(k), [] )

True

## Conclusion Ineq5

In [54]:
def ineq5(k): 
    return NS(x(k+1) - x_star) + 1/2 * NS(x(k+1) - x(k)) + 2 * lamda(k) * (1 + theta(k)) * ( f(x(k)) - f_star )     \
        <= NS(x(k) - x_star) + 1/2 * NS(x(k) - x(k-1)) + 2 * lamda(k) * theta(k) * ( f(x(k-1)) - f_star )  

display( ineq5(k) )

0.5 || -x^{k} + x^{1+k} ||^2+2 ( - f(x^{\star})+f(x^{k}) ) ( 1+\theta_{k} ) \lambda_{k}+|| -x^{\star} + x^{1+k} ||^2 \le 0.5 || -x^{-1+k} + x^{k} ||^2+2 ( - f(x^{\star})+f(x^{-1+k}) ) \lambda_{k} \theta_{k}+|| -x^{\star} + x^{k} ||^2

In [55]:
add_of( ineq7(k), -NS(x(k) - x_star) + 2 * lamda(k) * ( f(x(k)) - f_star ) )

-|| -x^{\star} + x^{k} ||^2+2 ( - f(x^{\star})+f(x^{k}) ) \lambda_{k}+|| -x^{\star} + x^{1+k} ||^2 \le -|| -x^{\star} + x^{k} ||^2+|| -x^{\star} + x^{k} ||^2+|| -x^{k} + x^{1+k} ||^2

In [56]:
propadd_transivity( add_of( ineq7(k), -NS(x(k) - x_star) + 2 * lamda(k) * ( f(x(k)) - f_star ) ), ineq2(k) )

In [57]:
def ineq5_1(k): return transivity_eq_le_lt( add_of( ineq7(k), -NS(x(k) - x_star) + 2 * lamda(k) * ( f(x(k)) - f_star ) ), ineq2(k) )

display( ineq5_1(k) )

-|| -x^{\star} + x^{k} ||^2+2 ( - f(x^{\star})+f(x^{k}) ) \lambda_{k}+|| -x^{\star} + x^{1+k} ||^2 \le -0.5 || -x^{k} + x^{1+k} ||^2+0.5 || -x^{-1+k} + x^{k} ||^2+2 ( - f(x^{k})+f(x^{-1+k}) ) \lambda_{k} \theta_{k}

In [58]:
def ineq5_2(k): return add_of( ineq5_1(k), NS( x(k) - x_star ) + 1/2 * NS( x(k+1) - x(k) ) + 2 * lamda(k) * theta(k) * ( f(x(k)) - f_star ) )

display( ineq5_2(k) )

-|| -x^{\star} + x^{k} ||^2+0.5 || -x^{k} + x^{1+k} ||^2+2 ( - f(x^{\star})+f(x^{k}) ) \lambda_{k}+2 ( - f(x^{\star})+f(x^{k}) ) \lambda_{k} \theta_{k}+|| -x^{\star} + x^{1+k} ||^2+|| -x^{\star} + x^{k} ||^2 \le 0.5 || -x^{-1+k} + x^{k} ||^2+2 ( - f(x^{\star})+f(x^{k}) ) \lambda_{k} \theta_{k}+2 ( - f(x^{k})+f(x^{-1+k}) ) \lambda_{k} \theta_{k}+|| -x^{\star} + x^{k} ||^2

In [59]:
verify( ineq5(k), [] )

True

# Tests

In [60]:
lamda(k).is_nonnegative = True
theta(k).is_nonnegative = True
(lamda(k) * theta(k)).is_nonnegative

'unknown'

In [61]:
lamda(k).is_nonnegative = True
lamda(k).is_nonnegative

'unknown'

In [62]:
a = lamda(k)
b = lamda(k)
a.is_nonnegative = True
print( b.is_nonnegative )
print( a==b )

unknown
True
